# LoRA fine-tuning of Qwen3-VL-4B-Instruct on ROCOv2

Notebook counterpart of `qwen3vl_roco_finetune.py` (use the `.py` under `tmux` for the
real run; this notebook is for inspecting the setup and running the **probe**).

### Design, and where it comes from
Follows the two papers closest to this task --- the [ROCOv2 LoRA study](https://pmc.ncbi.nlm.nih.gov/articles/PMC12730038/)
(same dataset, same task) and [Swin-Qwen3](https://www.frontiersin.org/journals/radiology/articles/10.3389/fradi.2026.1875905/full)
(Front. Radiol. 2026). Both use:

* a **single training stage** (not multi-stage),
* a **frozen vision encoder**,
* **LoRA on the LLM + the multimodal projector** (their `mm_proj` = our `visual.merger`),
* **bf16 / fp16, no quantization**, lr **1e-4**.

Qwen3-VL's own report (Table 1) trains the merger alone in Stage 0 only to *bootstrap*
a freshly-paired ViT+LLM. Our model is **already aligned** (zero-shot BERTScore 0.6539
with no training at all), so that stage does not transfer --- hence one stage here.

| | Trainable | LR |
|---|---|---|
| LoRA r=16, &alpha;=32 on LLM `q,k,v,o_proj` + `gate,up,down_proj` | yes | 1e-4 |
| `visual.merger` + `visual.deepstack_merger_list.{0,1,2}` | yes | 2e-5 |
| ViT blocks, patch_embed, embeddings, lm_head | **frozen** | --- |

Freezing the ViT means its activations are never stored, which is what keeps this
inside 12 GB in **bf16 with no quantization** --- avoiding both the bitsandbytes
CC 7.0 risk on Volta and the dequantization slowdown.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import json, math, random, time
import torch
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if device == "cuda" else torch.float32
print("device =", device, "| dtype =", DTYPE)

In [ ]:
# ------------------------------- Config ------------------------------- #
MODEL_ID     = "/home/matei/qwen3-vl-4b-instruct"
RUN_TAG      = "qwen3vl_ft"
OUT_DIR      = f"/home/matei/{RUN_TAG}_ckpt"

# 12,000 subsample = the size used for the BLIP-2 / BioMedVQA fine-tunes.
# 768 visual tokens = the V1 zero-shot eval setting, so the delta needs no baseline re-run.
N_TRAIN      = 12000
N_VAL        = 500
SEED         = 42
MIN_PIXELS   = 256 * 32 * 32
MAX_PIXELS   = 768 * 32 * 32
MAX_CAP_TOK  = 48                 # ROCO captions average ~21 words

EPOCHS       = 2
BATCH_SIZE   = 1                  # 12 GB -> 1 (same finding as the zero-shot runs)
GRAD_ACCUM   = 4                  # effective batch 4 (Swin-Qwen3 used exactly 1x4)
LR_LORA      = 1e-4               # both comparable papers use 1e-4 on this task
LR_MERGER    = 2e-5               # differential LR, mirroring the LLaVA recipe
WEIGHT_DECAY = 0.01               # both papers
WARMUP_FRAC  = 0.03
MAX_GRAD_NORM= 1.0
TRAIN_MERGER = True
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVAL_EVERY   = 500                # optimiser steps
PATIENCE     = 3                  # early stop on val loss (ROCOv2 paper: patience 3-6)
PROBE_STEPS  = 20

In [ ]:
# ---------------- The V1 prompt (identical to the zero-shot eval script) ---------------- #
SYSTEM_PROMPT = (
    "You are an expert radiologist writing captions for a radiology teaching archive. "
    "You are given one medical image. Write a single concise caption that describes only "
    "what is directly visible.\n"
    "Rules:\n"
    "1. Describe only what is observable in this image: the imaging modality, the anatomical "
    "region, and any clearly visible findings. Do not infer diagnoses, patient history, or "
    "findings that are not directly visible.\n"
    "2. If a finding cannot be determined from the image, do not state it.\n"
    "3. Be terse and clinical: one sentence, radiology-report style.\n"
    "4. Output only the caption -- no preamble, no \"This image shows\", no disclaimers."
)
USER_PROMPT = "Provide the caption for this image."

In [ ]:
# ---------------------- Load model + processor ---------------------- #
from transformers import AutoProcessor, AutoModelForImageTextToText, get_cosine_schedule_with_warmup

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"    # training (generation uses left; batch=1 anyway)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map={"": 0} if device == "cuda" else None)
model.config.use_cache = False                # incompatible with gradient checkpointing
print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ------------- Freeze everything, then re-enable only LoRA + the mergers ------------- #
for p in model.parameters():
    p.requires_grad = False

# `visual.merger` (main) + the three DeepStack mergers that feed LLM layers 1-3.
# This is the vision->language bridge the ROCOv2 paper adapts as "mm_proj".
MERGER_PREFIXES = ("model.visual.merger", "model.visual.deepstack_merger_list")
merger_params = []
if TRAIN_MERGER:
    for n, p in model.named_parameters():
        if n.startswith(MERGER_PREFIXES):
            p.requires_grad = True
            p.data = p.data.float()           # fp32 master weights -> stable AdamW updates
            merger_params.append(p)

# LoRA on the LLM only: these suffixes exist solely under model.language_model.*
# (the ViT uses qkv/proj/linear_fc1/2, so there is no accidental match in the vision tower).
from peft import LoraConfig, get_peft_model
lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
# PEFT's _mark_only_adapters_as_trainable() re-freezes EVERY non-LoRA parameter,
# including the mergers unfrozen above. Re-assert them -- merger_params holds the same
# Parameter objects, so this restores the flags on the modules actually in the graph.
for p in merger_params:
    p.requires_grad = True
assert all(p.requires_grad for p in merger_params), "merger re-freeze not repaired"
for n, p in model.named_parameters():
    if "lora_" in n:
        p.data = p.data.float()
        p.requires_grad = True

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()            # else no grad flows through checkpointed blocks

lora_params = [p for n, p in model.named_parameters() if "lora_" in n and p.requires_grad]
n_lora, n_merger = sum(p.numel() for p in lora_params), sum(p.numel() for p in merger_params)
n_total = sum(p.numel() for p in model.parameters())
print(f"trainable: LoRA {n_lora/1e6:.1f}M + merger {n_merger/1e6:.1f}M "
      f"= {(n_lora+n_merger)/1e6:.1f}M / {n_total/1e9:.2f}B "
      f"({100*(n_lora+n_merger)/n_total:.2f}%)")

In [ ]:
# ------------------------------ Data ------------------------------ #
import pandas as pd
ROCO_DIR = "/home/matei/rocov2"

def load_split(split, csv):
    img_dir = os.path.join(ROCO_DIR, split)
    caps = pd.read_csv(os.path.join(ROCO_DIR, csv)).dropna(subset=["Caption"]).reset_index(drop=True)
    return [{"id": r.ID, "path": os.path.join(img_dir, f"{r.ID}.jpg"), "caption": str(r.Caption)}
            for r in caps.itertuples() if os.path.isfile(os.path.join(img_dir, f"{r.ID}.jpg"))]

train_all, val_all = load_split("train", "train_captions.csv"), load_split("valid", "valid_captions.csv")
rng = random.Random(SEED)                      # fixed seed -> reproducible subsample
rng.shuffle(train_all); rng.shuffle(val_all)
train_recs, val_recs = train_all[:N_TRAIN], val_all[:N_VAL]
print(f"train {len(train_recs)} (of {len(train_all)}) | val {len(val_recs)}")

In [ ]:
# ------------- Example construction: loss masked to the CAPTION only ------------- #
_DUMMY = Image.new("RGB", (32, 32))
_msgs = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [{"type": "image", "image": _DUMMY},
                                   {"type": "text",  "text": USER_PROMPT}]},
]
PROMPT_TEXT = processor.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
EOS = processor.tokenizer.eos_token or "<|im_end|>"

def build_example(rec):
    """Prompt+caption tokenised; labels = -100 everywhere except the caption tokens."""
    img = Image.open(rec["path"]).convert("RGB")          # radiographs -> RGB
    cap = " ".join(str(rec["caption"]).split())
    cap_ids = processor.tokenizer(cap, add_special_tokens=False)["input_ids"][:Ò]
    cap = processor.tokenizer.decode(cap_ids)             # truncate to the token budget
    full = processor(text=[PROMPT_TEXT + cap + EOS], images=[img], return_tensors="pt")
    # Prompt length WITH the same image, so the expanded <|image_pad|> seats are counted.
    plen = processor(text=[PROMPT_TEXT], images=[img], return_tensors="pt")["input_ids"].shape[1]
    labels = full["input_ids"].clone()
    labels[:, :plen] = -100                               # mask prompt/system/image tokens
    labels[full["attention_mask"] == 0] = -100            # mask padding
    full["labels"] = labels
    return full

# Sanity: only the caption should be supervised.
ex = build_example(train_recs[0]); lab = ex["labels"][0]
print(f"seq={lab.numel()} tokens | supervised={(lab != -100).sum().item()}")
print("supervised text ->", processor.tokenizer.decode(lab[lab != -100])[:90])

In [ ]:
# ------------------- Optimiser: two groups (LoRA fast, merger slow) ------------------- #
groups = [{"params": lora_params, "lr": LR_LORA, "weight_decay": WEIGHT_DECAY}]
if merger_params:
    groups.append({"params": merger_params, "lr": LR_MERGER, "weight_decay": WEIGHT_DECAY})
optim = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)

steps_per_epoch = max(1, math.ceil(len(train_recs) / (BATCH_SIZE * GRAD_ACCUM)))
total_steps     = max(1, int(steps_per_epoch * EPOCHS))
sched = get_cosine_schedule_with_warmup(optim, int(WARMUP_FRAC * total_steps), total_steps)
print(f"{steps_per_epoch} optimiser steps/epoch x {EPOCHS} epochs = {total_steps} total")

def forward_loss(rec):
    ex = {k: v.to(model.device) for k, v in build_example(rec).items()}
    with torch.autocast("cuda", dtype=DTYPE):
        return model(**ex).loss

In [ ]:
# ------------- PROBE: measure peak VRAM and s/step before committing days ------------- #
# Run this BEFORE the full training. If it OOMs, lower MAX_PIXELS (768 -> 512) and re-run.
model.train()
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
for i in tqdm(range(PROBE_STEPS), desc="probe"):
    loss = forward_loss(train_recs[i]) / GRAD_ACCUM
    loss.backward()
    if (i + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(lora_params + merger_params, MAX_GRAD_NORM)
        optim.step()
        sched.step()
        optim.zero_grad(set_to_none=True)

dt   = (time.time() - t0) / PROBE_STEPS
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"\nMAX_VIS_TOKENS : {MAX_PIXELS // (32*32)}")
print(f"peak VRAM      : {peak:.2f} GB / 12.6 GB")
print(f"s/sample       : {dt:.2f}")
print(f"-> 1 epoch over {len(train_recs)} imgs = {dt*len(train_recs)/3600:.1f} h")
print(f"-> {EPOCHS} epochs = {dt*len(train_recs)*EPOCHS/3600:.1f} h")

### Next: run the real training headless

The full run belongs in `tmux`, not the notebook:

```bash
cd /home/matei
PROBE=1 miniconda3/envs/vlm/bin/python qwen3vl_roco_finetune.py      # confirm VRAM + s/step

tmux new -d -s qwen_ft \
  'cd /home/matei && miniconda3/envs/vlm/bin/python qwen3vl_roco_finetune.py 2>&1 | tee ft_run.log'
```

It checkpoints `best` (lowest validation loss) and `last`, early-stops on patience 3,
and `RESUME=1` continues an interrupted run.

Then score the fine-tuned model **through the same eval script as the zero-shot run**,
so the delta is clean:

```bash
LORA_PATH=/home/matei/qwen3vl_ft_ckpt/best RUN_TAG=qwen3vl_ft_test \
  miniconda3/envs/vlm/bin/python qwen3vl_roco_zeroshot_v1.py
```

Compare against **zero-shot V1 = 0.6539** BERTScore, and against the current best
system, **LLaVA ViT-1 = 0.6708**.